# Fly-In: Drone Simulation Challenge Maps

This collection contains carefully crafted maps designed to test different aspects of drone pathfinding algorithms, from basic navigation to complex optimization challenges.

## Map Categories

### 🟢 Easy Maps
**Target**: Beginners, basic algorithm testing
- `01_linear_path.txt` - Simple linear progression (2 drones)
- `02_simple_fork.txt` - Basic path splitting (4 drones)  
- `03_basic_capacity.txt` - Introduction to capacity constraints (4 drones)

### 🟡 Medium Maps
**Target**: Intermediate challenges, algorithm optimization
- `01_dead_end_trap.txt` - Dead ends that can trap naive algorithms (5 drones)
- `02_circular_loop.txt` - Circular paths with restricted zones (6 drones)
- `03_priority_puzzle.txt` - Optimal path selection with priority zones (5 drones)

### 🔴 Hard Maps
**Target**: Advanced algorithms, stress testing
- `01_maze_nightmare.txt` - Complex maze with multiple traps and loops (8 drones)
- `02_capacity_hell.txt` - Extreme capacity constraints requiring careful timing (12 drones)
- `03_ultimate_challenge.txt` - **THE ULTIMATE TEST** - All challenges combined (15 drones)

### ⚫ Challenger Maps
**Target**: Research and algorithmic limits exploration
- `01_the_impossible_dream.txt` - **THE IMPOSSIBLE DREAM** - Quasi-unsolvable challenge (25 drones)

> ⚠️ **WARNING**: Challenger maps are designed to push algorithmic limits and may not be solvable by most implementations. They are intended for research, stress testing, and algorithmic exploration rather than validation. The goal is to challenge the boundaries of what's possible, not to pass evaluation criteria.

> 🏆 **CHALLENGE RECORD**: The reference implementation solves "The Impossible Dream" in **45 turns**. Can you beat this record? This serves as a benchmark for algorithmic optimization and provides a concrete goal for advanced implementations.

## Challenge Types Covered

### 🎯 **Dead End Traps**
Maps contain paths that lead nowhere, testing if algorithms can backtrack or avoid getting stuck.

### 🔄 **Circular Loops** 
Cycles in the graph that can cause infinite loops in poorly designed algorithms.

### ⚡ **Capacity Constraints**
- Zone capacity limits (max_drones)
- Connection capacity limits (max_link_capacity)
- Timing-based bottlenecks

### 🚀 **Zone Type Optimization**
- `normal`: Standard 1-turn movement
- `restricted`: 2-turn movement (slow but sometimes necessary)
- `priority`: 1-turn movement but should be preferred
- `blocked`: Completely inaccessible

### 🧩 **Complex Topology**
- Multiple valid paths with different costs
- Convergence points requiring coordination
- Multi-layer challenges

## Testing Strategy

1. **Start with Easy**: Ensure basic functionality works
2. **Progress to Medium**: Test algorithm robustness
3. **Challenge with Hard**: Stress test optimization and edge cases
4. **Ultimate Test**: `03_ultimate_challenge.txt` combines all difficulties

## Expected Behavior

All maps are designed to be solvable with a well-implemented algorithm. However:

- **Easy maps**: Should solve quickly with any reasonable approach
- **Medium maps**: May require backtracking, path optimization, or capacity management
- **Hard maps**: Demand sophisticated algorithms with proper conflict resolution and optimization

## Performance Benchmarks

- **Easy**: < 10 simulation turns typically
- **Medium**: 10-30 simulation turns depending on optimization
- **Hard**: 30+ simulation turns, focus on finding valid solutions
- **Challenger**: **Record to beat: 45 turns** for "The Impossible Dream" - designed for algorithmic research


In [1]:
import sys


class ParsingError(Exception):
    def __init__(self, message: str) -> None:
        super().__init__(message)


print("Parsing...")
route = "maps/not_valid/"
maps = [
    "00_valid",
    "01_empty",
    "02_no_double_points",
    "03_negative_drones",
    "04_zero_drones",
    "05_too_many_drones",
    "06_no_start_hub",
    "07_no_end_hub",
    "08_same_names",
    "09_same_place",
    "10_invalid_node_name ",
    "11_invalid_node_name-",
    "12_invalid_connection",
    "13_duplicated_connection",
    ["14_a_invalid_metadata_zone", "14_b_invalid_metadata_zone"],
    ["15_a_invalid_metadata_connection", "15_b_invalid_metadata_connection"],
    "16_invalid_zone_type",
    "17_invalid_max_drones",
    "18_invalid_max_link",
    "19_max_drones_start",
    "20_max_drones_end",
    "21_no_coordinates",
    "22_one_coordinate",
    "23_no_metadata",
    "24_forbidden",
    "25_no_road",
    "26_unexpected_line",
    "27_blocked_exit"
]

try:
    #with open("maps/not_valid/" + maps[0] + ".txt", 'r') as f:
    with open("maps/easy/02_simple_fork.txt") as f:
        txt = f.read().strip()
        lines = txt.split("\n")
        while "" in lines:
            lines.remove("")
        valid = [i.split(':') for i in lines if i.find('#') != 0]
except FileNotFoundError as e:
    print(e)
    sys.exit()
except PermissionError as e:
    print(e)
    sys.exit()

try:
    for i in valid:
        if i[0].lower() == "nb_drones":
            break
    else:
        raise ParsingError("Number of drones is not defined")
    for i in valid:
        if i[0].find("hub") >= 0:
            break
    else:
        raise ParsingError("No hubs in the map")
    for i in valid:
        if i[0].lower() == "connection":
            break
    else:
        raise ParsingError("No connections in the map")
except ParsingError as e:
    print(e)
    sys.exit()

hubs, connections = [], []
try:
    for i in valid:
        if i[0].lower() == "nb_drones":
            nb = int(i[1])
            assert nb > 0 and nb < 1000
        elif i[0].lower().find("hub") >= 0:
            lst = i[1].strip().split(' ')
            node = {
                "id": lst[0],
                "x": int(lst[1]),
                "y": int(lst[2]),
                "attr": ' '.join(lst[3:]).lstrip('[').rstrip(']')
            }
            if len(node["attr"]) > 1:
                attr = node.get("attr").split(' ')
                for i in attr:
                    a = i.split('=')
                    node.update({a[0]: a[1]})
            node.pop("attr")
            if node["id"].find("-") >= 0:
                raise ParsingError(f"Invalid name for zone: {node['id']}")
            hubs.append(node)
        elif i[0].lower() == "connection":
            lst = i[1].strip().split(' ')
            nodes = lst[0].strip().split('-')
            if nodes[0] == nodes[1]:
                raise ParsingError(f"Connection with itself: {nodes[0]}")
            connection = {
                "node1": nodes[0],
                "node2": nodes[1],
                "attr": ' '.join(lst[1:]).lstrip('[').rstrip(']')
            }
            if len(connection["attr"]) > 1:
                attr = connection.get("attr").split(' ')
                for i in attr:
                    a = i.split('=')
                    connection.update({a[0]: a[1]})
            connection.pop("attr")
            ids = [i.get("id") for i in hubs]
            if connection["node1"] not in ids:
                raise ParsingError(f"Hub {connection['node1']} not defined")
            if connection["node2"] not in ids:
                raise ParsingError(f"Hub {connection['node2']} not defined")
            if connection in connections or {
                "node1": connection["node2"], "node2": connection["node1"]
            } in connections:
                raise ParsingError(f"Duplicated connection: {connection}")
            connections.append(connection)
    for i in hubs:
        if i["id"].find("start") >= 0:
            break
    else:
        raise ParsingError("No start hub")
    for i in hubs:
        if i["id"].find("goal") >= 0:
            break
    else:
        raise ParsingError("No goal hub")
    for i in hubs:
        for j in hubs[hubs.index(i):]:
            if i["id"] == j["id"] and i is not j:
                raise ValueError(f"Hubs with identical name: {i['id']}")
            if i["id"] == j["id"] and i is not j:
                raise ValueError(f"Hubs with identical name: {i['id']}")
except ValueError as e:
    print(e)
except IndexError:
    print("Invalid variable separation")
except AssertionError:
    print("Invalid number of drones")
except ParsingError as e:
    print(e)
else:
    print("Number of drones:", nb)
    print("Nodes:")
    for i in hubs:
        print(i)
    print("Connections:")
    for i in connections:
        print(i)

Parsing...
Number of drones: 4
Nodes:
{'id': 'start', 'x': 0, 'y': 0, 'color': 'green'}
{'id': 'junction', 'x': 1, 'y': 0, 'color': 'yellow', 'max_drones': '2'}
{'id': 'path_a', 'x': 2, 'y': 1, 'color': 'blue'}
{'id': 'path_b', 'x': 2, 'y': -1, 'color': 'blue'}
{'id': 'goal', 'x': 3, 'y': 0, 'color': 'red'}
Connections:
{'node1': 'start', 'node2': 'junction', 'max_link_capacity': '2'}
{'node1': 'junction', 'node2': 'path_a'}
{'node1': 'junction', 'node2': 'path_b'}
{'node1': 'path_a', 'node2': 'goal'}
{'node1': 'path_b', 'node2': 'goal'}


In [2]:
def check_hubs(hubs: dict) -> None:
    for node in hubs:
        attr = node.get("attr")
        if attr is None:
            return
        attrs = attr.split(' ')
        data = [i.split('=') for i in attrs]
        metadata = {i[0]: i[1] for i in data}

In [3]:
class Hub:
    def __init__(self, info: dict):
        self._id = info["id"]
        self._coords = (info["x"], info["y"])
        #attr = info["attr"].split(' ')
        #data = [i.split('=') for i in attr]
        #self._metadata = {i[0]: i[1] for i in data}
        self._color = info.get("color", "white")
        self._zone = info.get("zone", "normal")
        assert self._zone.lower() in ("normal", "blocked", "restricted", "priority")
        if self._id not in ("start", "goal"):
            self._max_drones = int(info.get("max_drones", 1))
            assert self._max_drones > 0
        self._neighbours: list[Node] = []
        self._g = 0
        self._h = 0
        self._f = 0
        self._occupation: int = 0

    @property
    def g(self) -> int:
        return self._g

    @g.setter
    def g(self, g: int) -> None:
        self._g = g

    @property
    def h(self) -> int:
        return self._h

    @h.setter
    def h(self, h: int) -> None:
        self._h = h

    @property
    def f(self) -> int:
        return self._f

    @f.setter
    def f(self, f: int) -> None:
        self._f = f

    @property
    def occupation(self) -> int:
        return self._occupation

    @occupation.setter
    def occupation(self, drones: int) -> None:
        self._occupation = drones


class Start(Hub):
    def __init__(self, info: dict) -> None:
        super().self.__init__(info)
        assert self._id.find("start") >= 0
        self._max_drones = -1
        self._zone = "normal"


class Goal(Hub):
    def __init__(self, info: dict) -> None:
        super().self.__init__(info)
        assert self._id.find("goal") >= 0
        self._max_drones = -1
        self._zone = "normal"


class Connection:
    def __init__(self, info: dict):
        self._connects = (info["node1"], info["node2"])
        if self._connects[0] == self._connects[1]:
            raise ValueError("Connection joins node with itself")
        attr = info.get("attr", "").split(' ')
        data = [i.split('=') for i in attr if len(i) == 2]
        if len(data) != 0:
            self._metadata = {i[0]: i[1] for i in data}
            self._max_link_capacity = self._metadata.get("max_link_capacity", 1)
            assert self._max_link_capacity > 0
        self._transiting: int = 0
        hubs: None | tuple[Hub, Hub] = None

    def get_hubs(self, hubs: list[Hub]) -> None:
        for hub in hubs:
            if hub._id == self._connects[0]:
                hub1 = hub
                break
        else:
            raise AttributeError("Hub 1 not found")
        for hub in hubs:
            if hub._id == self._connects[1]:
                hub2 = hub
                break
        else:
            raise AttributeError("Hub 2 not found")
        self._hubs = (hub1, hub2)

    @property
    def transiting(self) -> int:
        return self._transiting

    @transiting.setter
    def transiting(self, drones: int) -> None:
        self._transiting = drones

In [4]:
class Map:
    def __init__(
        self,
        drones: int,
        hubs: list[Hub],
        connections: list[Connection]
    ) -> None:
        self._num = drones
        self._hubs = hubs
        self._ids = [hub._id for hub in hubs]
        self._connections = connections
        for name in self._ids:
            if name.lower().find("start") >= 0:
                self._start = self._hubs[self._ids.index(name)]
                break
        else:
            raise ValueError("Start hub not found")
        for name in self._ids:
            if name.lower().find("goal") >= 0:
                self._goal = self._hubs[self._ids.index(name)]
                break
        else:
            raise ValueError("Goal hub not found")
        for i in self._connections:
            if i._connects[0] in self._ids and i._connects[1] in self._ids:
                a, b = i._connects
                hub_a = self._hubs[self._get_hub_index(a)]
                hub_b = self._hubs[self._get_hub_index(b)]
                hub_a._neighbours.append(hub_b)
                hub_b._neighbours.append(hub_a)
            else:
                raise ValueError(f"Invalid connection: {i}")

    def add_hub(self, hub: Hub) -> None:
        if hub._id not in [hub._id for hub in self._hubs]:
            self._hubs.append(hub)

    def _get_hub_index(self, hub_name: str) -> int:
        for i in range(len(self._hubs)):
            if self._hubs[i]._id == hub_name:
                return i
        else:
            raise ValueError("Hub not in map")

    def add_connection(self, connection: Connection) -> None:
        a, b = connection._connects
        if connection._id not in [way._id for way in self._connections]:
            if False not in [hub._id in self._hubs for hub in (a, b)]:
                self._connections.append(connection)
                hub_a = self._hubs[self._get_hub_index(a._id)]
                hub_b = self._hubs[self._get_hub_index(b._id)]
                hub_a._neighbours.append(hub_b)
                hub_b._neighbours.append(hub_a)


hub_list = [Hub(hub) for hub in hubs]
connection_list = [Connection(connection) for connection in connections]
level = Map(4, hub_list, connection_list)
#print(map._ids)
for hub in level._hubs:
    print(f"{hub._id}: {[i._id for i in hub._neighbours]}")
for connection in level._connections:
    print(f"{connection._connects[0]}-{connection._connects[1]}")

start: ['junction']
junction: ['start', 'path_a', 'path_b']
path_a: ['junction', 'goal']
path_b: ['junction', 'goal']
goal: ['path_a', 'path_b']
start-junction
junction-path_a
junction-path_b
path_a-goal
path_b-goal


In [5]:
class Drone:
    def __init__(self, start: Start) -> None:
        self._current: Node = start
        self._goingto: Node | None = None
        self._transiting: Connection | None = None

In [6]:
class Simulation:
    def __init__(self, map: Map, drones: int, start: Hub = None, goal: Hub = None) -> None:
        self._map = map
        self._drones = [Drone]
        self._start = self._map._hubs